# Intron AfriSpeech-200 Automatic Speech Recognition Challenge
**Objective**: Create an automatic speech recognition (ASR) model for African accents.
**Metric**: Word Error Rate (WER)

In [ ]:
import os
import torch
import torchaudio
import pandas as pd
from datasets import load_dataset, load_metric
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, Trainer, TrainingArguments
import numpy as np
import re
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Data Preparation

In [ ]:
def remove_special_characters(batch):
    batch["transcription"] = re.sub(r'[\,\.\!\?\-\;\:\"\(\)]', '', batch["transcription"]).lower() + " "
    return batch

def speech_file_to_array_fn(batch):
    speech_array, sampling_rate = torchaudio.load(batch["path"])
    batch["speech"] = speech_array[0].numpy()
    batch["sampling_rate"] = sampling_rate
    batch["target_text"] = batch["transcription"]
    return batch

def prepare_dataset(batch):
    batch["input_values"] = processor(batch["speech"], sampling_rate=batch["sampling_rate"]).input_values[0]
    
    with processor.as_target_processor():
        batch["labels"] = processor(batch["target_text"]).input_ids
    return batch

## 2. Model and Trainer Setup

In [ ]:
MODEL_ID = "facebook/wav2vec2-large-xlsr-53"

try:
    processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
    model = Wav2Vec2ForCTC.from_pretrained(
        MODEL_ID, 
        attention_dropout=0.1,
        hidden_dropout=0.1,
        feat_proj_dropout=0.0,
        mask_time_prob=0.05,
        layerdrop=0.1,
        gradient_checkpointing=True, 
        ctc_loss_reduction="mean", 
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer)
    ).to(device)
except Exception as e:
    print(f"Error loading model: {e}")

## 3. Evaluation Function

In [ ]:
wer_metric = load_metric("wer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

## 4. Training (Example)

In [ ]:
training_args = TrainingArguments(
  output_dir="./wav2vec2-afrispeech",
  group_by_length=True,
  per_device_train_batch_size=8,
  gradient_accumulation_steps=2,
  evaluation_strategy="steps",
  num_train_epochs=30,
  fp16=True,
  save_steps=500,
  eval_steps=500,
  logging_steps=500,
  learning_rate=3e-4,
  warmup_steps=500,
  save_total_limit=2,
)

print("Training arguments defined. To start training, you would instantiate the Trainer and call .train().")